# LeetCode 202 — Happy Number bằng Python

**Link đề:** https://leetcode.com/problems/happy-number/
**Độ khó:** Easy · **Chủ đề:** Hash Table, Math, Two Pointers, Floyd's Cycle Finding

```python
class Solution:
    def isHappy(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Tách chữ số bằng `% 10` và `// 10`, tính tổng bình phương chữ số.
2. Nhận ra bài toán thực chất là **phát hiện chu trình** (cycle detection) trong một dãy số.
3. Dùng `set` để nhớ các giá trị đã gặp (cách 1), và **Floyd rùa–thỏ** với bộ nhớ `O(1)` (cách 2).
4. Hiểu vì sao dãy **không thể tăng mãi** — lý do khiến thuật toán chắc chắn dừng.
5. Biết mẹo toán học: mọi số không vui đều rơi vào chu trình chứa `4` (cách 3).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — số "vui" là gì?

Một số `n` là **happy number** nếu lặp lại quá trình sau:
- thay `n` bằng **tổng bình phương các chữ số** của nó,
- lặp cho tới khi `n == 1` (khi đó dừng mãi ở 1), **hoặc** dãy lặp vô hạn trong một vòng không chứa 1.

Nếu dãy đi tới 1 → trả `True`, ngược lại `False`.

**Ví dụ 1:** `n = 19` → `True`

```text
1² + 9² = 82
8² + 2² = 68
6² + 8² = 100
1² + 0² + 0² = 1   -> vui!
```

**Ví dụ 2:** `n = 2` → `False` (dãy chạy mãi trong vòng `4 → 16 → 37 → 58 → 89 → 145 → 42 → 20 → 4`).

**Constraints:** `1 <= n <= 2^31 - 1`.

Câu hỏi cần trả lời: Làm sao biết "lặp vô hạn" khi máy tính không thể chạy vô hạn?

## 2. Nền tảng Python: tách chữ số bằng `% 10` và `// 10`

- `x % 10` → chữ số **cuối** (phần dư khi chia 10).
- `x // 10` → bỏ chữ số cuối (chia lấy phần nguyên).
- `d ** 2` hoặc `d * d` → bình phương.

Ví dụ `x = 82`: `82 % 10 = 2`, `82 // 10 = 8`, `8 % 10 = 8`, `8 // 10 = 0` → dừng.

In [1]:
x = 82
while x > 0:
    d = x % 10
    print(f"x={x:>3} -> chữ số cuối d={d}, d**2={d**2}, x // 10 = {x // 10}")
    x //= 10
print("Phân biệt: 82 / 10 =", 82 / 10, "| 82 // 10 =", 82 // 10)

x= 82 -> chữ số cuối d=2, d**2=4, x // 10 = 8
x=  8 -> chữ số cuối d=8, d**2=64, x // 10 = 0
Phân biệt: 82 / 10 = 8.2 | 82 // 10 = 8


### 2.1. Hàm `next_number` — một bước của quá trình

Đây là "viên gạch" duy nhất của bài: từ `n` sinh ra số tiếp theo.

In [2]:
def next_number(n: int) -> int:
    total = 0
    while n > 0:
        d = n % 10
        total += d * d
        n //= 10
    return total

for n in [19, 82, 68, 100, 2, 4]:
    print(n, "->", next_number(n))

19 -> 82
82 -> 68
68 -> 100
100 -> 1
2 -> 4
4 -> 16


### 2.2. Cách viết khác bằng chuỗi + generator

`str(n)` biến số thành chuỗi chữ số, `int(ch)` đổi từng ký tự về số. `sum(... for ch in s)` là **generator expression** — một vòng `for` gói gọn trong `sum`.

In [3]:
def next_number_str(n: int) -> int:
    return sum(int(ch) ** 2 for ch in str(n))

print(next_number_str(19), next_number_str(2))
assert all(next_number(k) == next_number_str(k) for k in range(1, 10000))
print("Hai cách tính next_number cho cùng kết quả trên 1..9999")

82 4
Hai cách tính next_number cho cùng kết quả trên 1..9999


## 3. Quan sát bằng tay — in dãy số

Ta thử in dãy của `19` và của `2` (giới hạn 20 bước để không chạy vô hạn).

In [4]:
def sequence(n: int, steps: int = 20) -> list[int]:
    seq = [n]
    for _ in range(steps):
        n = next_number(n)
        seq.append(n)
        if n == 1:
            break
    return seq

print("19:", sequence(19))
print(" 2:", sequence(2))
print(" 7:", sequence(7))

19: [19, 82, 68, 100, 1]
 2: [2, 4, 16, 37, 58, 89, 145, 42, 20, 4, 16, 37, 58, 89, 145, 42, 20, 4, 16, 37, 58]
 7: [7, 49, 97, 130, 10, 1]


Nhận xét:
- Dãy của `19` chạm 1.
- Dãy của `2` lặp lại `4, 16, 37, 58, 89, 145, 42, 20, 4, ...` — một **chu trình** (cycle).
- Khi một giá trị **xuất hiện lần thứ hai**, mọi thứ phía sau sẽ lặp y hệt (vì `next_number` là hàm xác định: cùng đầu vào → cùng đầu ra).

⇒ "Lặp vô hạn" ⇔ "gặp lại một số đã thấy".

## 4. Cách trực giác (baseline) — giới hạn số bước

Người mới hay nghĩ: "cứ lặp 1000 lần, nếu chưa thấy 1 thì coi như không vui".

Cách này **chạy được** trong thực tế nhưng không có lý do chắc chắn: tại sao 1000 mà không phải 100? Nếu có số nào cần 1001 bước thì sao? Ta cần một điều kiện dừng có **chứng minh**.

In [5]:
def is_happy_limit(n: int, limit: int = 1000) -> bool:
    for _ in range(limit):
        if n == 1:
            return True
        n = next_number(n)
    return n == 1

print(is_happy_limit(19), is_happy_limit(2))

True False


## 5. Nút thắt & insight — đây là bài phát hiện chu trình

Nút thắt: cần biết **khi nào** dừng với `False`.

Insight 1: Dừng khi gặp **số đã thấy** → cần cấu trúc dữ liệu nhớ "đã thấy" và tra cứu nhanh → `set`.

Insight 2 (vì sao chắc chắn dừng): dãy **không thể tăng mãi**.
- Số có `d` chữ số thì tổng bình phương chữ số tối đa là `81·d` (mỗi chữ số tối đa `9² = 81`).
- Với `n <= 2^31 - 1` (10 chữ số): `next <= 810`.
- Với `n <= 999`: `next <= 81·3 = 243`.
- Với số có `d >= 4` chữ số: `n >= 10^(d-1) > 81·d`, nên `next < n` — dãy giảm cho tới khi xuống dưới 1000.

Vậy sau vài bước, mọi giá trị đều nằm trong `[1, 999]` (thật ra `<= 243`). Chỉ có hữu hạn giá trị → chắc chắn sẽ lặp lại hoặc chạm 1.

In [6]:
print("Lớn nhất của next_number trên 1..999:", max(next_number(k) for k in range(1, 1000)))
print("next_number(2**31 - 1) =", next_number(2**31 - 1))
print("next_number(9_999_999_999) =", next_number(9_999_999_999), "= 81*10")
# Kiểm tra: với mọi số >= 1000 (thử mẫu), next < n
import random
rng = random.Random(202)
assert all(next_number(k) < k for k in (rng.randint(1000, 2**31 - 1) for _ in range(5000)))
print("Với 5000 số ngẫu nhiên >= 1000: luôn có next < n")

Lớn nhất của next_number trên 1..999: 243
next_number(2**31 - 1) = 260
next_number(9_999_999_999) = 810 = 81*10
Với 5000 số ngẫu nhiên >= 1000: luôn có next < n


## 6. Nền tảng: `set` — tập hợp, tra cứu nhanh

`set` lưu các phần tử **không trùng**, không có thứ tự. Thao tác chính:

| Thao tác | Cú pháp | Ý nghĩa |
|---|---|---|
| Tạo rỗng | `seen = set()` | (chú ý: `{}` là **dict** rỗng, không phải set) |
| Thêm | `seen.add(x)` | thêm `x`; thêm lại lần nữa không đổi gì |
| Kiểm tra | `x in seen` | trung bình `O(1)` nhờ băm (hash) |
| Kích thước | `len(seen)` | số phần tử |

In [7]:
seen = set()
seen.add(19)
seen.add(82)
seen.add(19)               # trùng -> không thêm
print(seen, "len =", len(seen))
print("82 in seen?", 82 in seen, "| 68 in seen?", 68 in seen)
print("type({}) =", type({}), "| type(set()) =", type(set()))

{82, 19} len = 2
82 in seen? True | 68 in seen? False
type({}) = <class 'dict'> | type(set()) = <class 'set'>


## 7. Cách 1 — Lời giải chính với `set`

Từng đoạn:

**Đoạn 1:** tập `seen` rỗng.
**Đoạn 2:** lặp khi `n != 1` **và** `n` chưa từng thấy:
```python
while n != 1 and n not in seen:
```
**Đoạn 3:** ghi nhận `n` rồi sang số tiếp theo:
```python
    seen.add(n)
    n = next_number(n)
```
**Đoạn 4:** vòng lặp dừng vì một trong hai lý do. Nếu dừng vì `n == 1` → vui; nếu dừng vì `n in seen` → chu trình → không vui:
```python
return n == 1
```

In [8]:
def is_happy_set(n: int) -> bool:
    seen = set()
    while n != 1 and n not in seen:
        seen.add(n)
        n = next_number(n)
    return n == 1

for n in [19, 2, 7, 1, 4]:
    print(n, "->", is_happy_set(n))

19 -> True
2 -> False
7 -> True
1 -> True
4 -> False


## 8. TRACE cách 1

### 8.1. Bảng trace `n = 19`

| Bước | `n` đầu bước | `n == 1`? | `n in seen`? | Hành động | `seen` sau |
|---:|---:|:---:|:---:|---|---|
| 1 | 19 | không | không | thêm 19, n = 82 | {19} |
| 2 | 82 | không | không | thêm 82, n = 68 | {19, 82} |
| 3 | 68 | không | không | thêm 68, n = 100 | {19, 82, 68} |
| 4 | 100 | không | không | thêm 100, n = 1 | {19, 82, 68, 100} |
| 5 | 1 | **có** | — | dừng, trả `True` | |

In [9]:
def trace_happy_set(n: int) -> bool:
    seen = set()
    step = 1
    while n != 1 and n not in seen:
        nxt = next_number(n)
        print(f"Bước {step}: n={n:>3} chưa thấy -> thêm vào seen, next={nxt}")
        seen.add(n)
        n = nxt
        step += 1
    reason = "n == 1" if n == 1 else f"{n} đã có trong seen (chu trình)"
    print(f"Dừng vì {reason}. Kết quả: {n == 1}. Kích thước seen = {len(seen)}")
    return n == 1

trace_happy_set(19)

Bước 1: n= 19 chưa thấy -> thêm vào seen, next=82
Bước 2: n= 82 chưa thấy -> thêm vào seen, next=68
Bước 3: n= 68 chưa thấy -> thêm vào seen, next=100
Bước 4: n=100 chưa thấy -> thêm vào seen, next=1
Dừng vì n == 1. Kết quả: True. Kích thước seen = 4


True

### 8.2. Trace nhánh "chu trình" — `n = 2`

Đây là nhánh quan trọng mà ví dụ 1 không cho thấy: dừng vì gặp lại `4`.

In [10]:
trace_happy_set(2)

Bước 1: n=  2 chưa thấy -> thêm vào seen, next=4
Bước 2: n=  4 chưa thấy -> thêm vào seen, next=16
Bước 3: n= 16 chưa thấy -> thêm vào seen, next=37
Bước 4: n= 37 chưa thấy -> thêm vào seen, next=58
Bước 5: n= 58 chưa thấy -> thêm vào seen, next=89
Bước 6: n= 89 chưa thấy -> thêm vào seen, next=145
Bước 7: n=145 chưa thấy -> thêm vào seen, next=42
Bước 8: n= 42 chưa thấy -> thêm vào seen, next=20
Bước 9: n= 20 chưa thấy -> thêm vào seen, next=4
Dừng vì 4 đã có trong seen (chu trình). Kết quả: False. Kích thước seen = 9


False

## 9. Cách 2 — Floyd rùa–thỏ (bộ nhớ O(1))

Câu hỏi: có phát hiện chu trình mà **không cần nhớ** các giá trị đã gặp không?

Hình dung dãy số như một đường chạy có thể vòng lại. Thả hai con trỏ:
- **slow** (rùa): mỗi lượt đi 1 bước: `slow = next(slow)`.
- **fast** (thỏ): mỗi lượt đi 2 bước: `fast = next(next(fast))`.

Nếu dãy chạm 1 → `fast` tới 1 trước (và 1 → 1 mãi). Nếu có chu trình không chứa 1 → khi cả hai vào vòng, thỏ mỗi lượt rút ngắn khoảng cách thêm 1 bước, nên **chắc chắn đuổi kịp** rùa: `slow == fast`.

Ta dừng khi `fast == 1` (vui) hoặc `slow == fast` (gặp nhau trong vòng không có 1).

Đây là kỹ thuật giống LeetCode 141 Linked List Cycle — chỉ khác "node tiếp theo" là `next_number(n)`.

In [11]:
def is_happy_floyd(n: int) -> bool:
    slow = n
    fast = next_number(n)
    while fast != 1 and slow != fast:
        slow = next_number(slow)                 # 1 bước
        fast = next_number(next_number(fast))    # 2 bước
    return fast == 1

for n in [19, 2, 7, 1, 4]:
    print(n, "->", is_happy_floyd(n))

19 -> True
2 -> False
7 -> True
1 -> True
4 -> False


### 9.1. Vì sao khởi tạo `fast = next_number(n)` chứ không phải `fast = n`?

Nếu `slow = fast = n` thì điều kiện `slow != fast` sai ngay từ đầu → vòng lặp không chạy. Cho thỏ đi trước 1 bước là cách đơn giản nhất. (Cách khác: dùng `while True` và kiểm tra sau khi di chuyển.)

Trường hợp `n = 1`: `fast = next_number(1) = 1` → dừng ngay, trả `True`. Đúng.

In [12]:
def trace_happy_floyd(n: int) -> bool:
    slow, fast = n, next_number(n)
    step = 0
    print(f"Khởi tạo: slow={slow}, fast={fast}")
    while fast != 1 and slow != fast:
        slow = next_number(slow)
        fast = next_number(next_number(fast))
        step += 1
        print(f"Lượt {step}: slow={slow:>3}, fast={fast:>3}")
    print("Dừng:", "fast == 1 -> vui" if fast == 1 else f"slow == fast == {slow} -> chu trình")
    return fast == 1

trace_happy_floyd(19)
print()
trace_happy_floyd(2)

Khởi tạo: slow=19, fast=82
Lượt 1: slow= 82, fast=100
Lượt 2: slow= 68, fast=  1
Dừng: fast == 1 -> vui

Khởi tạo: slow=2, fast=4
Lượt 1: slow=  4, fast= 37
Lượt 2: slow= 16, fast= 89
Lượt 3: slow= 37, fast= 42
Lượt 4: slow= 58, fast=  4
Lượt 5: slow= 89, fast= 37
Lượt 6: slow=145, fast= 89
Lượt 7: slow= 42, fast= 42
Dừng: slow == fast == 42 -> chu trình


False

## 10. Cách 3 — Mẹo toán học: mọi chu trình "không vui" đều chứa 4

Người ta đã kiểm chứng (chỉ cần thử các số `1..243` vì mọi dãy đều rơi vào khoảng này): chỉ có **hai** trạng thái kết thúc:
- điểm cố định `1`,
- chu trình `4 → 16 → 37 → 58 → 89 → 145 → 42 → 20 → 4`.

Vậy ta chỉ cần lặp tới khi `n == 1` hoặc `n == 4`. Cell dưới **tự chứng minh bằng máy** cho các số `1..999`, không cần tin suông.

In [13]:
def find_cycle_values(limit: int = 999) -> set[int]:
    cycle_values = set()
    for start in range(1, limit + 1):
        seen_order = []
        seen = set()
        n = start
        while n not in seen:
            seen.add(n)
            seen_order.append(n)
            n = next_number(n)
        # n là giá trị đầu tiên bị lặp -> phần từ n trở đi là chu trình
        idx = seen_order.index(n)
        cycle_values |= set(seen_order[idx:])
    return cycle_values

print("Mọi giá trị nằm trong các chu trình (1..999):", sorted(find_cycle_values()))

Mọi giá trị nằm trong các chu trình (1..999): [1, 4, 16, 20, 37, 42, 58, 89, 145]


In [14]:
def is_happy_math(n: int) -> bool:
    while n != 1 and n != 4:
        n = next_number(n)
    return n == 1

print([k for k in range(1, 50) if is_happy_math(k)])

[1, 7, 10, 13, 19, 23, 28, 31, 32, 44, 49]


Giải thích output: tập chu trình chỉ gồm `{1}` (điểm cố định) và tám số `{4, 16, 20, 37, 42, 58, 89, 145}`. Vậy mọi dãy không vui chắc chắn đi qua `4`. Cách 3 ngắn gọn nhưng **dựa vào một sự thật đặc thù** của bài; trong phỏng vấn nên trình bày cách 1/2 trước, rồi nhắc cách 3 như kiến thức thêm.

## 11. Code nộp LeetCode

Bản chính dùng `set` (dễ giải thích nhất):

In [15]:
class Solution:
    def isHappy(self, n: int) -> bool:
        def next_number(x: int) -> int:
            total = 0
            while x > 0:
                d = x % 10
                total += d * d
                x //= 10
            return total

        seen = set()
        while n != 1 and n not in seen:
            seen.add(n)
            n = next_number(n)
        return n == 1

sol = Solution()
print(sol.isHappy(19), sol.isHappy(2))

True False


Bản Floyd (bộ nhớ `O(1)`) — cũng nộp được:

In [16]:
class SolutionFloyd:
    def isHappy(self, n: int) -> bool:
        def nxt(x: int) -> int:
            total = 0
            while x:
                x, d = divmod(x, 10)     # divmod trả (x // 10, x % 10) cùng lúc
                total += d * d
            return total

        slow, fast = n, nxt(n)
        while fast != 1 and slow != fast:
            slow = nxt(slow)
            fast = nxt(nxt(fast))
        return fast == 1

print(divmod(82, 10))
print(SolutionFloyd().isHappy(19), SolutionFloyd().isHappy(2))

(8, 2)
True False


## 12. Vì sao đúng?

**Cách 1 (set)** — invariant ở đầu mỗi vòng: `seen` chứa đúng các số đã xuất hiện **trước** `n` trong dãy, và chưa số nào bằng 1.
- Nếu `n == 1`: dãy chạm 1 → vui. Đúng.
- Nếu `n in seen`: `n` xuất hiện lần 2. Vì `next_number` xác định, đoạn từ lần đầu tới lần hai sẽ lặp mãi; đoạn đó không chứa 1 (nếu chứa 1 thì ta đã dừng trước) → không vui. Đúng.
- **Dừng chắc chắn:** mục 5 cho thấy dãy rơi vào tập hữu hạn `[1, 999]`, nên không thể có vô hạn số khác nhau → sớm muộn sẽ lặp.

**Cách 2 (Floyd)** — `fast` luôn đi trước `slow`. Nếu có chu trình độ dài `L` không chứa 1: khi cả hai trong vòng, mỗi lượt khoảng cách (tính theo chiều đi) giảm đúng 1 → sau tối đa `L` lượt sẽ bằng 0 → gặp nhau. Nếu dãy chạm 1, `fast` tới 1 trước hoặc cùng lúc `slow`, ta trả `True`. Hai con trỏ không thể gặp nhau ở số khác 1 khi dãy chạm 1, vì một khi tới 1 thì dãy đứng yên ở 1.

## 13. Time / Space và bảng so sánh

Gọi `d` = số chữ số của `n` (`d <= 10`). Một lần `next_number` tốn `O(d)`. Sau bước đầu, mọi giá trị `<= 810`, sau bước nữa `<= 243` — nên số bước trước khi lặp bị chặn bởi một hằng số (≈ vài chục). Do đó:

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Baseline giới hạn 1000 bước | `O(1000·d)` | `O(1)` | không có lý do chắc chắn |
| 1. `set` | `O(log n)` (bước đầu) + hằng số | `O(1)` thực tế (≤ ~20 phần tử) | dễ hiểu, dễ trace |
| 2. Floyd | như trên, ~gấp 3 lần gọi `next` | `O(1)` | không cần cấu trúc phụ |
| 3. Dừng ở 4 | như trên | `O(1)` | dựa vào sự thật đặc thù |

Theo cách viết sách giáo khoa: Time `O(log n)` (do số chữ số `≈ log10 n`), Space `O(log n)` cho set (thận trọng) hoặc `O(1)` cho Floyd.

In [17]:
# Đếm thực nghiệm số bước tối đa trước khi dừng (cách set) trên 1..100000
def steps_set(n: int) -> int:
    seen = set()
    steps = 0
    while n != 1 and n not in seen:
        seen.add(n)
        n = next_number(n)
        steps += 1
    return steps

worst = max(range(1, 100001), key=steps_set)
print("Số bước nhiều nhất trên 1..100000:", steps_set(worst), "tại n =", worst)

Số bước nhiều nhất trên 1..100000: 20 tại n = 15999


## 14. Bộ test

1. Ví dụ đề: `19 → True`, `2 → False`.
2. Biên: `1 → True` (đã là 1), `2^31 - 1`.
3. Số nằm sẵn trong chu trình: `4`, `16`, `89` → `False`.
4. Số vui khác: `7, 10, 100, 1111111` → `True`.
5. Đối chiếu 4 cách trên `1..20000` + số ngẫu nhiên có seed.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("ví dụ 1 (19)", sol.isHappy(19), True)
check("ví dụ 2 (2)", sol.isHappy(2), False)
check("biên n=1", sol.isHappy(1), True)
check("trong chu trình 4", sol.isHappy(4), False)
check("trong chu trình 89", sol.isHappy(89), False)
check("số vui 7", sol.isHappy(7), True)
check("số vui 10", sol.isHappy(10), True)
check("1111111 (7 chữ số 1 -> 7 -> vui)", sol.isHappy(1111111), True)
check("lớn nhất 2^31-1", sol.isHappy(2**31 - 1), is_happy_set(2**31 - 1))

PASS ví dụ 1 (19): True
PASS ví dụ 2 (2): False
PASS biên n=1: True
PASS trong chu trình 4: False
PASS trong chu trình 89: False
PASS số vui 7: True
PASS số vui 10: True
PASS 1111111 (7 chữ số 1 -> 7 -> vui): True
PASS lớn nhất 2^31-1: False


In [19]:
import random
fns = [is_happy_limit, is_happy_set, is_happy_floyd, is_happy_math,
       Solution().isHappy, SolutionFloyd().isHappy]
for n in range(1, 20001):
    assert len({f(n) for f in fns}) == 1, n
rng = random.Random(2020)
for _ in range(3000):
    n = rng.randint(1, 2**31 - 1)
    assert len({f(n) for f in fns}) == 1, n
print("6 cài đặt khớp nhau trên 23000 input. Số vui trong 1..100:",
      [k for k in range(1, 101) if is_happy_set(k)])

6 cài đặt khớp nhau trên 23000 input. Số vui trong 1..100: [1, 7, 10, 13, 19, 23, 28, 31, 32, 44, 49, 68, 70, 79, 82, 86, 91, 94, 97, 100]


## 15. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `while True` không có điều kiện thoát cho chu trình

```python
while n != 1:
    n = next_number(n)    # n = 2 -> chạy mãi mãi, notebook treo
return True
```
Sửa: thêm `n not in seen` (cách 1) hoặc Floyd.

### Lỗi 2 — Thêm `n` vào `seen` **sau khi** đã đổi `n`

```python
while n != 1 and n not in seen:
    n = next_number(n)
    seen.add(n)           # SAI: vừa thêm n mới, vòng sau kiểm tra 'n in seen' thấy ngay -> dừng sớm
```
Với `19`: `n = 82`, thêm 82, kiểm tra `82 in seen` → `True` → trả `False`. Sai!

In [20]:
def is_happy_bug_order(n: int) -> bool:
    seen = set()
    while n != 1 and n not in seen:
        n = next_number(n)
        seen.add(n)       # thứ tự sai
    return n == 1

print("Bản lỗi với 19:", is_happy_bug_order(19), "(đúng phải là True)")
print("Bản đúng với 19:", is_happy_set(19))

Bản lỗi với 19: False (đúng phải là True)
Bản đúng với 19: True


### Lỗi 3 — Dùng `/` thay `//` khi tách chữ số

`n = n / 10` biến `n` thành số thực `8.2, 0.82, ...` — vòng `while n > 0` gần như không dừng (chỉ dừng khi số thực bị làm tròn về 0 sau hàng trăm bước) và `n % 10` cho số thực sai.

### Lỗi 4 — Floyd khởi tạo `slow = fast = n`

Điều kiện `slow != fast` sai ngay → vòng không chạy → trả `fast == 1` tức `n == 1`. Với `19` sẽ trả `False`. Sai!

In [21]:
def is_happy_floyd_bug(n: int) -> bool:
    slow = fast = n
    while fast != 1 and slow != fast:
        slow = next_number(slow)
        fast = next_number(next_number(fast))
    return fast == 1

print("Floyd lỗi khởi tạo, n=19:", is_happy_floyd_bug(19), "(đúng phải là True)")

Floyd lỗi khởi tạo, n=19: False (đúng phải là True)


### Lỗi 5 — Dùng `{}` để tạo set rỗng

`seen = {}` tạo **dict**. `seen.add(n)` sẽ báo `AttributeError: 'dict' object has no attribute 'add'`. Dùng `set()`.

In [22]:
seen = {}
try:
    seen.add(5)
except AttributeError as e:
    print("Lỗi:", e)

Lỗi: 'dict' object has no attribute 'add'


## 16. Mẹo nhận dạng pattern

- Một quá trình **lặp hàm** `x → f(x)` với hỏi "có dừng ở X không" / "có lặp vô hạn không" → **phát hiện chu trình**.
- Hai công cụ: **set "đã thấy"** (dễ, tốn bộ nhớ) và **Floyd rùa–thỏ** (bộ nhớ `O(1)`).
- Bài cùng họ: 141/142 Linked List Cycle, 287 Find the Duplicate Number (mảng như "danh sách liên kết" `i → nums[i]`), 457 Circular Array Loop.
- Xử lý chữ số: `% 10`, `// 10`, `divmod` — giống 7 Reverse Integer, 9 Palindrome Number, 258 Add Digits.
- Khi muốn chứng minh "dãy không chạy ra vô cực": chặn trên bằng số chữ số (`81·d`).

## 17. Biến thể

### 17.1. Trả về độ dài chu trình hoặc các số trong chu trình
Với `n` không vui, trả list các số trong chu trình theo thứ tự gặp. Dùng `dict` lưu `số → vị trí xuất hiện` thay vì `set`.

### 17.2. Tổng **lập phương** chữ số (cube)
Thay `d*d` bằng `d**3`. Các điểm cố định nổi tiếng: `153 = 1³+5³+3³`, `370`, `371`, `407` (số Armstrong 3 chữ số). Cùng khung phát hiện chu trình vẫn dùng được.

In [23]:
def cycle_of(n: int, power: int = 2) -> list[int]:
    def f(x):
        return sum(int(c) ** power for c in str(x))
    pos = {}          # số -> vị trí lần đầu gặp
    order = []
    while n not in pos:
        pos[n] = len(order)
        order.append(n)
        n = f(n)
    return order[pos[n]:]

print("Chu trình của 2 (bình phương):", cycle_of(2))
print("Chu trình của 19 (bình phương):", cycle_of(19))
print("Chu trình của 153 (lập phương):", cycle_of(153, 3))
print("Chu trình của 4 (lập phương):", cycle_of(4, 3))

Chu trình của 2 (bình phương): [4, 16, 37, 58, 89, 145, 42, 20]
Chu trình của 19 (bình phương): [1]
Chu trình của 153 (lập phương): [153]
Chu trình của 4 (lập phương): [133, 55, 250]


## 18. Bài tập tự luyện

### Bài 1 — Tính tay
Tính dãy của `n = 7` cho tới khi dừng. Số đó có vui không? Mất bao nhiêu bước?

### Bài 2 — Dự đoán output
`next_number(0)`, `next_number(10)`, `next_number(999)` lần lượt bằng bao nhiêu?

### Bài 3 — TODO: viết hàm tổng bình phương chữ số

In [24]:
def exercise_next(n: int) -> int:
    # TODO: tách từng chữ số bằng % 10 và // 10, cộng bình phương
    pass

# assert exercise_next(19) == 82
# assert exercise_next(100) == 1

### Bài 4 — TODO: viết `isHappy` bằng set

In [25]:
def exercise_is_happy_set(n: int) -> bool:
    # TODO 1: seen = set()
    # TODO 2: lặp khi n != 1 và n chưa thấy: thêm n, đổi n
    # TODO 3: trả n == 1
    pass

# assert exercise_is_happy_set(19) is True
# assert exercise_is_happy_set(2) is False

### Bài 5 — TODO: viết bằng Floyd (không dùng set/list/dict)

In [26]:
def exercise_is_happy_floyd(n: int) -> bool:
    # TODO: slow đi 1 bước, fast đi 2 bước
    pass

# assert exercise_is_happy_floyd(7) is True
# assert exercise_is_happy_floyd(4) is False

### Bài 6 — Mini challenge: số vui thứ k
Viết `kth_happy(k)` trả về số vui thứ `k` (đếm từ 1): `kth_happy(1) = 1`, `kth_happy(2) = 7`, `kth_happy(5) = 19`. Gợi ý thêm: lưu kết quả `next_number` đã tính (memo) để tăng tốc khi `k` lớn.

In [27]:
def exercise_kth_happy(k: int) -> int:
    # TODO: duyệt n = 1, 2, 3, ... đếm số vui
    pass

# assert exercise_kth_happy(1) == 1
# assert exercise_kth_happy(5) == 19

## Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Áp công thức liên tục | `7 → 49` | `49 → 97 → 130 → 10 → 1` |
| 2 | Không có chữ số nào thì tổng = 0 | `while n > 0` không chạy khi `n = 0` | `10 → 1`, `999 → 243` |
| 3 | Giống đảo số | `d = n % 10; n //= 10` | `total += d * d` |
| 4 | Dừng khi gặp lại | `while n != 1 and n not in seen:` | `seen.add(n); n = next(n)` |
| 5 | Thỏ đi trước 1 bước | `slow, fast = n, next(n)` | `fast = next(next(fast))` |
| 6 | Đếm khi gặp số vui | `count += 1` khi vui | `if count == k: return n` |

## Đáp án tham khảo

**Bài 1:** `7 → 49 → 97 → 130 → 10 → 1`. Vui, mất 5 bước.

**Bài 2:** `next_number(0) = 0` (vòng `while` không chạy), `next_number(10) = 1`, `next_number(999) = 243`.

In [28]:
def answer_next(n: int) -> int:
    total = 0
    while n > 0:
        d = n % 10
        total += d * d
        n //= 10
    return total

def answer_is_happy_set(n: int) -> bool:
    seen = set()
    while n != 1 and n not in seen:
        seen.add(n)
        n = answer_next(n)
    return n == 1

def answer_is_happy_floyd(n: int) -> bool:
    slow, fast = n, answer_next(n)
    while fast != 1 and slow != fast:
        slow = answer_next(slow)
        fast = answer_next(answer_next(fast))
    return fast == 1

def answer_kth_happy(k: int) -> int:
    count = 0
    n = 0
    while count < k:
        n += 1
        if answer_is_happy_set(n):
            count += 1
    return n

assert sequence(7) == [7, 49, 97, 130, 10, 1]
assert answer_next(0) == 0 and answer_next(10) == 1 and answer_next(999) == 243
assert answer_next(19) == 82 and answer_next(100) == 1
assert answer_is_happy_set(19) is True and answer_is_happy_set(2) is False
assert answer_is_happy_floyd(7) is True and answer_is_happy_floyd(4) is False
assert answer_kth_happy(1) == 1 and answer_kth_happy(2) == 7 and answer_kth_happy(5) == 19
print("10 số vui đầu tiên:", [answer_kth_happy(k) for k in range(1, 11)])
print("Đáp án tham khảo PASS toàn bộ!")

10 số vui đầu tiên: [1, 7, 10, 13, 19, 23, 28, 31, 32, 44]
Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
# Phát hiện chu trình bằng set           # Floyd rùa–thỏ
seen = set()                             slow, fast = x, f(x)
while x != TARGET and x not in seen:     while fast != TARGET and slow != fast:
    seen.add(x)                              slow = f(slow)
    x = f(x)                                 fast = f(f(fast))
return x == TARGET                       return fast == TARGET
```

**Checklist trước khi nộp:**
- [ ] Tên method `isHappy`, trả `bool`.
- [ ] Tách chữ số bằng `% 10` và `//` (không dùng `/`).
- [ ] Thêm vào `seen` **trước** khi đổi `n`.
- [ ] Floyd: `fast` bắt đầu đi trước 1 bước.
- [ ] Đã thử `1`, `2`, `4`, `19`, `7`.